# Imports

In [14]:
import os
import sys
import glob
sys.path.insert(0, "/project/def-nahee/kbas/Graphnet-Applications/Metadata")
import paths
from icecube import icetray

import pandas as pd
from collections import Counter
from icecube import dataio


from icecube import LeptonInjector, simclasses 



icetray.I3Logger.global_logger = icetray.I3NullLogger()

# Settings

In [15]:
DATASET = "STRING340MC"

# Available Datasets

In [16]:
all_datasets = {
    name: val for name, val in vars(paths).items()
    if not name.startswith("_") and isinstance(val, dict)
    and all(isinstance(v, dict) for v in val.values())
}

for ds_name, ds in all_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if isinstance(info, dict):
                status = info["path"] if info.get("path") else "None"
                print(f"[{ds_name}] {geometry}/{flavor}: {status}")

[LIC] STRING340MC/Muon: /project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Generator
[LIC] STRING340MC/Electron: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator
[LIC] STRING340MC/Tau: /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator
[LIC] STRING340MC/NC: /project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator
[LIC] SPRING2026MC/Muon: None
[LIC] SPRING2026MC/Electron: None
[LIC] SPRING2026MC/Tau: None
[LIC] SPRING2026MC/NC: None
[SPRING2026MC_I3] full_geometry/Muon: /project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator
[SPRING2026MC_I3] full_geometry/Electron: /project/6008051/pone_simulation/MC000009-nu_e-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator
[SPRING2026MC_I3] full_geometry/Tau: /project/6008051/pone_simulation/MC000010-nu_tau-2_6-LeptonInjector_PROPOSAL_clsim-

# Dataset Selection

In [17]:
selected_datasets = {
    name: val for name, val in all_datasets.items()
    if DATASET in name
}

print(f"Datasets matching '{DATASET}':")
for name in selected_datasets:
    print(f"  {name}")

Datasets matching 'STRING340MC':
  STRING340MC_I3
  STRING340MC_PMT
  STRING340MC_PARQUET
  STRING340MC_PARQUET_MIXED


In [18]:
selected_datasets = {
    name: val for name, val in selected_datasets.items()
    if "PMT" not in name and "PARQUET" not in name
}

print("After excluding PMT and PARQUET:")
for name in selected_datasets:
    print(f"  {name}")

After excluding PMT and PARQUET:
  STRING340MC_I3


In [19]:
print(f"Available (non-None) entries:")
for ds_name, ds in selected_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if isinstance(info, dict) and info.get("path") is not None:
                print(f"  [{ds_name}] {geometry}/{flavor}: {info['path']}  (format={info['format']})")

Available (non-None) entries:
  [STRING340MC_I3] full_geometry/Muon: /project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon  (format=i3)
  [STRING340MC_I3] full_geometry/Electron: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator  (format=zst)
  [STRING340MC_I3] full_geometry/Tau: /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator  (format=zst)
  [STRING340MC_I3] full_geometry/NC: /project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator  (format=zst)
  [STRING340MC_I3] 102_string/Muon: /home/kbas/scratch/String340MC/102_string/Muon_I3Photons  (format=gz)
  [STRING340MC_I3] 102_string/Electron: /home/kbas/scratch/String340MC/102_string/Electron_I3Photons  (format=gz)
  [STRING340MC_I3] 102_string/Tau: /home/kbas/scratch/String340MC/102_string/Tau_I3Photons  (format=gz)
  [STRING340MC_I3] 102_string/NC: /home/kbas/scrat

In [20]:
def i3_file_pattern(info):
    extension = info["format"]
    if extension == "i3":
        return "*.i3"
    return f"*.i3.{extension}"

print("File counts:")
for ds_name, ds in selected_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if isinstance(info, dict) and info.get("path") is not None:
                count = len(glob.glob(os.path.join(info["path"], "**", i3_file_pattern(info)), recursive=True))
                print(f"  [{ds_name}] {geometry}/{flavor}: {count} files")

File counts:
  [STRING340MC_I3] full_geometry/Muon: 9817 files
  [STRING340MC_I3] full_geometry/Electron: 6882 files
  [STRING340MC_I3] full_geometry/Tau: 9996 files
  [STRING340MC_I3] full_geometry/NC: 10000 files
  [STRING340MC_I3] 102_string/Muon: 9806 files
  [STRING340MC_I3] 102_string/Electron: 6867 files
  [STRING340MC_I3] 102_string/Tau: 9984 files
  [STRING340MC_I3] 102_string/NC: 9977 files


In [21]:
representative_files = {}

for ds_name, ds in selected_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if isinstance(info, dict) and info.get("path") is not None:
                files = sorted(glob.glob(os.path.join(info["path"], "**", i3_file_pattern(info)), recursive=True))
                key = f"{ds_name}/{geometry}/{flavor}"
                representative_files[key] = files[0] if files else None
                print(f"  {key}: {representative_files[key]}")

  STRING340MC_I3/full_geometry/Muon: /project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_000.i3
  STRING340MC_I3/full_geometry/Electron: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_000.i3.zst
  STRING340MC_I3/full_geometry/Tau: /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_000.i3.zst
  STRING340MC_I3/full_geometry/NC: /project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator/gen_000.i3.zst
  STRING340MC_I3/102_string/Muon: /home/kbas/scratch/String340MC/102_string/Muon_I3Photons/muon_cls_000.i3.gz
  STRING340MC_I3/102_string/Electron: /home/kbas/scratch/String340MC/102_string/Electron_I3Photons/electron_gen_000.i3.gz
  STRING340MC_I3/102_string/Tau: /home/kbas/scratch/String340MC/102_string/Tau_I3Photons/tau_gen_001.i3.gz
  STRING340MC_I3/102_string/NC: /home/kbas/scratch/String340MC/102_strin

# File Analysis

## Frame Types

In [22]:
rows = []
for label, path in representative_files.items():
    if path is None:
        continue
    counts = Counter()
    f = dataio.I3File(path)
    while f.more():
        frame = f.pop_frame()
        counts[str(frame.Stop)] += 1
    f.close()
    row = {"file": label}
    row.update(counts)
    rows.append(row)

df_frames = pd.DataFrame(rows).fillna(0).set_index("file")
df_frames = df_frames.astype(int)
df_frames

,TrayInfo,DAQ
file,,
STRING340MC_I3/full_geometry/Muon,1,200
STRING340MC_I3/full_geometry/Electron,1,200
STRING340MC_I3/full_geometry/Tau,1,200
STRING340MC_I3/full_geometry/NC,1,200
STRING340MC_I3/102_string/Muon,2,46
STRING340MC_I3/102_string/Electron,2,14
STRING340MC_I3/102_string/Tau,2,17
STRING340MC_I3/102_string/NC,2,26


## TrayInfo Frames

In [23]:
for label, path in representative_files.items():
    if path is None:
        continue
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    f = dataio.I3File(path)
    while f.more():
        frame = f.pop_frame()
        if frame.Stop == icetray.I3Frame.TrayInfo:
            for key in frame.keys():
                print(frame[key])
            break
    f.close()


STRING340MC_I3/full_geometry/Muon
============================= configuration ==============================
      Icetray run started:  Tue Mar 18 16:32:33 2025
                  on host:  cdr295.int.cedar.computecanada.ca (Linux-5.14.0-427.37.1.el9_4.x86_64/x86_64/gcc-11.4.0)
                  as user:  cmiller
          and gcc version:  11.4.0
                  svn url:  
             svn revision:  0

===========================   svn:externals   ============================

=============================   services    ==============================
=============================    modules    ==============================
reader (I3Reader)
  DropBuffers
    Description :  Tell I3Frames not to cache buffers of serialized frameobject data (this saves memory at the expense of processing speed and the ability to passthru unknown frame objects)
    Default     :  False
    Configured  :  None

  Filename
    Description :  Filename to read.  Use either this or Filenamelist, not both.

## Simulation Frames

In [24]:
for label, path in representative_files.items():
    if path is None:
        continue
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    f = dataio.I3File(path)
    sim_count = 0
    while f.more():
        frame = f.pop_frame()
        if str(frame.Stop) == "Simulation":
            sim_count += 1
            print(f"\n--- Simulation frame {sim_count} ---")
            for key in frame.keys():
                print(f"  {key}: {type(frame[key]).__name__}")
                print(f"    {frame[key]}")
    f.close()


STRING340MC_I3/full_geometry/Muon

STRING340MC_I3/full_geometry/Electron

STRING340MC_I3/full_geometry/Tau

STRING340MC_I3/full_geometry/NC

STRING340MC_I3/102_string/Muon

STRING340MC_I3/102_string/Electron

STRING340MC_I3/102_string/Tau

STRING340MC_I3/102_string/NC


## DAQ Frame Keys

In [25]:
daq_keys = {}

for label, path in representative_files.items():
    if path is None:
        continue
    f = dataio.I3File(path)
    while f.more():
        frame = f.pop_frame()
        if frame.Stop == icetray.I3Frame.DAQ:
            daq_keys[label] = set(frame.keys())
            break
    f.close()

all_keys = sorted(set(k for keys in daq_keys.values() for k in keys))
rows = []
for label, keys in daq_keys.items():
    row = {"file": label}
    for k in all_keys:
        row[k] = "✓" if k in keys else "✗"
    rows.append(row)

pd.DataFrame(rows).set_index("file")

,EventProperties,I3EventHeader,I3MCTree,I3MCTree_RNGState,I3MCTree_postprop,I3Photons,MMCTrackList
file,,,,,,,
STRING340MC_I3/full_geometry/Muon,✓,✓,✓,✓,✓,✓,✓
STRING340MC_I3/full_geometry/Electron,✓,✓,✓,✓,✓,✓,✓
STRING340MC_I3/full_geometry/Tau,✓,✓,✓,✓,✓,✓,✓
STRING340MC_I3/full_geometry/NC,✓,✓,✓,✓,✓,✓,✓
STRING340MC_I3/102_string/Muon,✓,✓,✓,✓,✓,✓,✓
STRING340MC_I3/102_string/Electron,✓,✓,✓,✓,✓,✓,✓
STRING340MC_I3/102_string/Tau,✓,✓,✓,✓,✓,✓,✓
STRING340MC_I3/102_string/NC,✓,✓,✓,✓,✓,✓,✓


## Print full geometry `I3MCTree_postprop`


In [26]:
for label, path in representative_files.items():
    if path is None or "/full_geometry/" not in label:
        continue

    print(f"\n{'=' * 80}")
    print(label)
    print(path)

    f = dataio.I3File(path)
    found = False

    while f.more():
        frame = f.pop_frame()
        if "I3MCTree_postprop" in frame:
            print(frame["I3MCTree_postprop"])
            found = True
            break

    f.close()

    if not found:
        print("I3MCTree_postprop not found")


STRING340MC_I3/full_geometry/Muon
/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_000.i3
[I3MCTree:
  13 NuMu (46.7688m, 344.607m, 276.395m) (122.007deg, 79.0101deg) 0ns 119.301GeV 0m Primary
    7 MuMinus (46.7688m, 344.607m, 276.395m) (122.952deg, 80.7891deg) 0ns 97.2484GeV 377.572m Dark
      17 MuMinus (46.7688m, 344.607m, 276.395m) (123.011deg, 80.8016deg) 0ns 97.2484GeV 19.2187m Null
      18 DeltaE (44.1951m, 328.693m, 286.858m) (123.011deg, 80.8016deg) 64.1067ns 0.77235GeV 19.2187m Null
      19 MuMinus (44.1951m, 328.693m, 286.858m) (122.996deg, 80.7741deg) 64.1067ns 91.808GeV 12.7465m Null
      20 DeltaE (42.4816m, 318.14m, 293.799m) (122.996deg, 80.7741deg) 106.624ns 1.85901GeV 31.9652m Null
      21 MuMinus (42.4816m, 318.14m, 293.799m) (122.877deg, 80.754deg) 106.624ns 86.8552GeV 15.7771m Null
      22 DeltaE (40.3511m, 305.072m, 302.379m) (122.877deg, 80.754deg) 159.251ns 0.977276GeV 47.7423m Null
      23 MuMinus (40.351

## Print full geometry `MMCTrackList`


In [28]:
for label, path in representative_files.items():
    if path is None or "/full_geometry/" not in label:
        continue

    print(f"\n{'=' * 80}")
    print(label)
    print(path)

    f = dataio.I3File(path)
    found = False

    while f.more():
        frame = f.pop_frame()
        if "MMCTrackList" in frame:
            print(frame["MMCTrackList"])
            found = True
            break

    f.close()

    if not found:
        print("MMCTrackList not found")



STRING340MC_I3/full_geometry/Muon
/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_000.i3
[I3MMCTrack = [
 (xi, yi, zi, ti, Ei) = (46.7688 ,344.607 ,276.395 ,0 ,97.2484)
 (xc, yc, zc, tc, Ec) = (27.8353 ,227.424 ,353.196 ,471.598 ,59.4044)
 (xf, yf, zf, tf, Ef) = (-2.50006 ,31.682 ,481.839 ,1261.47 ,-38942.5)
 Elost = 97.1428
 Particle = [ I3Particle MajorID : 3019218780441209185
             MinorID : 7
              Zenith : 2.14592
             Azimuth : 1.41004
                   X : 46.7688
                   Y : 344.607
                   Z : 276.395
                Time : 0
              Energy : 97.2484
               Speed : 0.299792
              Length : 377.572
                Type : MuMinus
        PDG encoding : 13
               Shape : MCTrack
              Status : NotSet
            Location : InIce
]]
]

STRING340MC_I3/full_geometry/Electron
/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_cls

## Find non-empty `MMCTrackList` in Tau files


In [29]:
for ds_name, ds in selected_datasets.items():
    for geometry, flavors in ds.items():
        for flavor, info in flavors.items():
            if flavor.lower() != "tau":
                continue
            if not isinstance(info, dict) or info.get("path") is None:
                continue

            label = f"{ds_name}/{geometry}/{flavor}"
            files = sorted(
                glob.glob(
                    os.path.join(info["path"], "**", i3_file_pattern(info)),
                    recursive=True,
                )
            )

            print(f"\n{'=' * 80}")
            print(label)
            print(f"{len(files)} files")

            found = False

            for file_index, file_path in enumerate(files, start=1):
                f = dataio.I3File(file_path)

                while f.more():
                    frame = f.pop_frame()
                    if "MMCTrackList" not in frame:
                        continue

                    mmc_tracks = frame["MMCTrackList"]
                    try:
                        n_tracks = len(mmc_tracks)
                    except TypeError:
                        n_tracks = None

                    if n_tracks is None or n_tracks > 0:
                        print(f"Found non-empty MMCTrackList in file {file_index}/{len(files)}")
                        print(file_path)
                        print(f"Frame stop: {frame.Stop}")
                        if n_tracks is not None:
                            print(f"Number of tracks: {n_tracks}")
                        print(mmc_tracks)
                        found = True
                        break

                f.close()

                if found:
                    break

            if not found:
                print("No non-empty MMCTrackList found")



STRING340MC_I3/full_geometry/Tau
9996 files
Found non-empty MMCTrackList in file 1/9996
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_000.i3.zst
Frame stop: DAQ
Number of tracks: 1
[I3MMCTrack = [
 (xi, yi, zi, ti, Ei) = (-212.866 ,-697.865 ,-17.8474 ,0 ,1092.36)
 (xc, yc, zc, tc, Ec) = (-212.869 ,-697.86 ,-17.8442 ,0.0214062 ,1.77686)
 (xf, yf, zf, tf, Ef) = (-212.869 ,-697.86 ,-17.8442 ,0.0214062 ,-143515)
 Elost = 1090.58
 Particle = [ I3Particle MajorID : 385304311471010140
             MinorID : 47
              Zenith : 2.09461
             Azimuth : 5.21737
                   X : -212.866
                   Y : -697.865
                   Z : -17.8474
                Time : 0
              Energy : 1092.36
               Speed : 0.299792
              Length : 0.00641741
                Type : TauMinus
        PDG encoding : 15
               Shape : MCTrack
              Status : NotSet
            Location : InIce
]]
]

S